In [1]:
from google.colab import userdata
from huggingface_hub import login
import sys
import os

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
!git clone https://$GITHUB_TOKEN@github.com/Constantine1824/TRI-AI-SLM.git
sys.path.append('/content/TRI-AI-SLM')
login(token=userdata.get('HF_TOKEN'))
%cd /content/TRI-AI-SLM
!git pull origin main

fatal: destination path 'TRI-AI-SLM' already exists and is not an empty directory.
/content/TRI-AI-SLM
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 518 bytes | 86.00 KiB/s, done.
From https://github.com/Constantine1824/TRI-AI-SLM
 * branch            main       -> FETCH_HEAD
   2b0951b..74f9f32  main       -> origin/main
Updating 2b0951b..74f9f32
Fast-forward
 finetune/config.py | 4 ++--
 utils/format.py    | 4 ++--
 2 files changed, 4 insertions(+), 4 deletions(-)


In [2]:
import importlib
importlib.invalidate_caches()

import importlib.util
print(importlib.util.find_spec('finetune'))

ModuleSpec(name='finetune', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bc76352b830>, origin='/content/TRI-AI-SLM/finetune/__init__.py', submodule_search_locations=['/content/TRI-AI-SLM/finetune'])


In [3]:
#!pip install trl peft bitsandbytes
import numpy as np
import pandas as pd
from finetune.trainer import collate_fn, finetune, evaluate
from utils.format import format_train_data, format_test_data

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [4]:
data = pd.read_csv('data/train_qa.csv')
data.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5


In [5]:
doc = pd.read_csv('data/documents.csv')
doc.head()

,document_id,title,topic,care_setting,population,text,origin,source_url,license
0,doc_chr_001,Type 2 diabetes self-management,chronic_disease,primary_care,adult,Type 2 diabetes management combines balanced m...,synthetic,NaN,CC0-1.0
1,doc_chr_002,Hypertension lifestyle measures,chronic_disease,primary_care,adult,"Lowering dietary salt, maintaining healthy wei...",synthetic,NaN,CC0-1.0
2,doc_chr_003,Asthma action plan basics,chronic_disease,primary_care,child,Children with asthma should use a written acti...,synthetic,NaN,CC0-1.0
3,doc_inf_001,Malaria prevention in endemic areas,infectious_disease,community,general,"Sleep under insecticide-treated nets, eliminat...",synthetic,NaN,CC0-1.0
4,doc_inf_002,Hand hygiene and respiratory etiquette,infectious_disease,community,general,Wash hands with soap for at least twenty secon...,synthetic,NaN,CC0-1.0


In [6]:
data = data.merge(doc[['document_id', 'text']], on='document_id', how='left')
data = data.rename(columns={'text':'context'})
data.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId,context
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1,"Lowering dietary salt, maintaining healthy wei..."
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2,Anaemia increases fatigue and adverse birth ou...
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3,Wash hands with soap for at least twenty secon...
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4,Type 2 diabetes management combines balanced m...
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5,Use weight-based paediatric paracetamol dosing...


In [7]:
from datasets import Dataset
test_data = pd.read_csv('data/test_questions.csv')

train_data = Dataset.from_pandas(data)
test_data = Dataset.from_pandas(test_data)
train_data = train_data.map(format_train_data)
test_data = test_data.map(format_test_data)

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [8]:
trainer = finetune(train_data)
eval = evaluate(trainer, test_data, batch_size=11)
eval

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
4,3.357086


Step,Training Loss
4,3.357086
8,2.226350
12,1.333787
16,0.781791
20,0.534158
24,0.412454
28,0.340774
32,0.267758
36,0.228518
40,0.191310


['\nAnswer:Iron-folic acid supplements, dietary counselling, and blood tests to check anaemia severity.',
 '\nAnswer:Check for breathing effort, mild doses of paracetamol if not contraindicated, and watch for lethargy.',
 '\nAnswer:Severe bleeding, high blood pressure, severe headache, or reduced fetal movement.',
 '\nAnswer:Ground rules, coping breathing, and connect with school support contacts.',
 '\nAnswer:Use insecticide-treated bed nets, indoor residual spraying, and take chemoprophylaxis as directed.',
 '\nAnswer:Use a spacer, reduce dust triggers, and consider reliever inhalers as directed.',
 '\nanswer:Half plate fruits/vegetables, one quarter protein, one quarter complex carbohydrates.',
 '\nAnswer:Complete the full course unless directed otherwise by a clinician.',
 '\nAnswer:Every 10 years, or after a dirty wound.',
 '\nAnswer:Signs of severe infection, dehydration, or diabetic ketoacidosis warrant urgent care.',
 '\nAnswer:Oral rehydration solution, separate toilet from be

In [11]:
test_df = pd.read_csv('data/test_questions.csv')
test_df.head()

,QuestionId,question,topic,care_setting,population
0,1001,How is pregnancy anaemia managed?,maternal_health,primary_care,pregnant
1,1002,Fever in a two-month-old — what to do?,emergency_triage,hospital,infant
2,1003,Pregnancy symptoms needing urgent review?,maternal_health,primary_care,pregnant
3,1004,Teen refuses school citing panic — approach?,mental_health_basics,school,child
4,1005,How can families prevent malaria?,infectious_disease,community,general


In [13]:
qid = test_df['QuestionId']
qid

,QuestionId
0,1001
1,1002
2,1003
3,1004
4,1005
5,1006
6,1007
7,1008
8,1009
9,1010


In [16]:
def save_submission(name, result, iteration=1):
  qid = test_df['QuestionId']
  cleaned = [s.lstrip('\n').removeprefix('Answer:').strip() for s in result]
  print(cleaned)
  qid['Answer'] = cleaned
  qid.to_csv(f'{name}-{iteration}')
  print(qid)
save_submission('submission.csv', eval)

['Iron-folic acid supplements, dietary counselling, and blood tests to check anaemia severity.', 'Check for breathing effort, mild doses of paracetamol if not contraindicated, and watch for lethargy.', 'Severe bleeding, high blood pressure, severe headache, or reduced fetal movement.', 'Ground rules, coping breathing, and connect with school support contacts.', 'Use insecticide-treated bed nets, indoor residual spraying, and take chemoprophylaxis as directed.', 'Use a spacer, reduce dust triggers, and consider reliever inhalers as directed.', 'answer:Half plate fruits/vegetables, one quarter protein, one quarter complex carbohydrates.', 'Complete the full course unless directed otherwise by a clinician.', 'Every 10 years, or after a dirty wound.', 'Signs of severe infection, dehydration, or diabetic ketoacidosis warrant urgent care.', 'Oral rehydration solution, separate toilet from bedroom, and call if high fever or blood in stool.']
0                                                  

/tmp/ipykernel_69872/3841300410.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qid['Answer'] = cleaned
